Risk Probability adalah probabilitas terjadinya gangguan rantai pasok yang diprediksi oleh model machine learning berdasarkan karakteristik transaksi logistik seperti shipping mode, lead time, scheduled delivery, dan wilayah distribusi.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path = "/content/drive/MyDrive/KuliahSLRPG/"

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
df = pd.read_csv(path+"DataCoSupplyChainDataset.csv", sep=',', encoding='latin-1')
df.head(10)
# sumber data https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis
# https://www.kaggle.com/code/gelarerouzbahani/data-analysis-for-supply-chain

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class
5,TRANSFER,6,4,18.580000,294.980011,Shipping canceled,0,73,Sporting Goods,Tonawanda,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/19/2018 11:03,Standard Class
6,DEBIT,2,1,95.180000,288.420013,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 10:42,First Class
7,TRANSFER,2,1,68.430000,285.140015,Late delivery,1,73,Sporting Goods,Miami,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 10:21,First Class
8,CASH,3,2,133.720001,278.589996,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 10:00,Second Class
9,CASH,2,1,132.149994,275.309998,Late delivery,1,73,Sporting Goods,San Ramon,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 9:39,First Class


In [8]:
df.shape

(180519, 53)

In [36]:
df['Shipping Mode'].value_counts()

,count
Shipping Mode,
Standard Class,107752
Second Class,35216
First Class,27814
Same Day,9737


In [37]:
numeric_cols = [
    'Days for shipping (real)',
    'Days for shipment (scheduled)'
]

df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Days for shipping (real),180519.0,3.497654,1.623722,0.0,2.0,3.0,5.0,6.0
Days for shipment (scheduled),180519.0,2.931847,1.374449,0.0,2.0,4.0,4.0,4.0


In [38]:
region_profile = pd.DataFrame({
    'Jumlah': df['Order Region'].value_counts(),
    'Persentase (%)': round(
        df['Order Region'].value_counts(normalize=True)*100,
        2
    )
})

region_profile.head(10)

,Jumlah,Persentase (%)
Order Region,,
Central America,28341,15.70
Western Europe,27109,15.02
South America,14935,8.27
Oceania,10148,5.62
Northern Europe,9792,5.42
Southeast Asia,9539,5.28
Southern Europe,9431,5.22
Caribbean,8318,4.61
West of USA,7993,4.43


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

cols = [
    'Shipping Mode',
    'Days for shipping (real)',
    'Days for shipment (scheduled)',
    'Order Region'
]

data = df[cols + ['Late_delivery_risk']].copy()

for col in ['Shipping Mode','Order Region']:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

X = data.drop('Late_delivery_risk', axis=1)
y = data['Late_delivery_risk']

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1
)

model.fit(X_train,y_train)

pred = model.predict(X_test)

print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           0       1.00      0.94      0.97     16307
           1       0.96      1.00      0.98     19797

    accuracy                           0.97     36104
   macro avg       0.98      0.97      0.97     36104
weighted avg       0.98      0.97      0.97     36104



In [30]:
proba = model.predict_proba(X_test)

risk_df = X_test.copy()

risk_df['Risk_Probability'] = proba[:,1]

top_risk = risk_df.sort_values(
    'Risk_Probability',
    ascending=False
).head(10)

print(top_risk)

        Shipping Mode  Days for shipping (real)  \
73943               1                         1   
48082               1                         1   
112670              1                         1   
112671              1                         1   
27244               1                         1   
19774               1                         1   
48141               1                         1   
36168               1                         1   
99043               1                         1   
112518              1                         1   

        Days for shipment (scheduled)  Order Region  Risk_Probability  
73943                               0             5          0.981862  
48082                               0             5          0.981862  
112670                              0             5          0.981862  
112671                              0             4          0.981862  
27244                               0             5          0.981862  
19774 

In [31]:
risk_prob = model.predict_proba(X_test)

risk_df = X_test.copy()

risk_df["Risk_Probability"] = risk_prob[:,1]

top_risk = (
    risk_df
    .sort_values(
        "Risk_Probability",
        ascending=False
    )
    .head(20)
)

top_risk

,Shipping Mode,Days for shipping (real),Days for shipment (scheduled),Order Region,Risk_Probability
73943,1,1,0,5,0.981862
48082,1,1,0,5,0.981862
112670,1,1,0,5,0.981862
112671,1,1,0,4,0.981862
27244,1,1,0,5,0.981862
19774,1,1,0,5,0.981862
48141,1,1,0,5,0.981862
36168,1,1,0,4,0.981862
99043,1,1,0,5,0.981862
112518,1,1,0,4,0.981862


# Supply Chain Risk Agent — Embedded Gradio App

Fully self-contained multi-agent assistant built on top of the XGBoost late-delivery risk model above: no `git clone`, no dependency on a GitHub repo being reachable. Every module's source code is embedded directly below and written to disk when you run the cells.


In [ ]:
# 1. Create a working directory
import os
os.makedirs("/content/supply_chain_risk_agent", exist_ok=True)
os.makedirs("/content/supply_chain_risk_agent/agents", exist_ok=True)
os.makedirs("/content/supply_chain_risk_agent/data", exist_ok=True)
%cd /content/supply_chain_risk_agent


In [ ]:
# 2. Export the trained model's real predictions so the agents can ground
# their answers in actual numbers instead of the LLM inventing them.
export_df = risk_df.copy()
export_df["Shipping Mode"] = df.loc[export_df.index, "Shipping Mode"]
export_df["Order Region"] = df.loc[export_df.index, "Order Region"]
export_df["Late_delivery_risk"] = y_test
export_df.to_csv("data/risk_predictions.csv", index=True, index_label="order_id")

region_risk = (
    export_df.groupby("Order Region")["Risk_Probability"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
region_risk.to_csv("data/region_risk_summary.csv", index=False)

mode_risk = (
    export_df.groupby("Shipping Mode")["Risk_Probability"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
mode_risk.to_csv("data/shipping_mode_risk_summary.csv", index=False)

print(f"Exported {len(export_df)} risk predictions, {len(region_risk)} regions, {len(mode_risk)} shipping modes")


In [ ]:
%%writefile requirements.txt
# Core LangChain ecosystem
langchain>=0.3.0,<0.4.0
langchain-core>=0.3.0,<0.4.0
langchain-community>=0.3.0,<0.4.0
langgraph>=0.2.0,<0.3.0
langsmith>=0.1.100,<0.2.0

# Groq LLM (free tier)
groq>=0.11.0

# Local embeddings (no API key needed)
sentence-transformers>=3.0.0

# Vector store
faiss-cpu>=1.8.0,<2.0.0

# UI
gradio

# Utilities
python-dotenv>=1.0.0
pydantic>=2.9.0,<3.0.0
pydantic-settings>=2.5.0
pandas>=2.0.0
numpy>=1.26.0,<2.0.0
SQLAlchemy>=2.0.0
grandalf


In [ ]:
# 3. Install dependencies
# pydantic<2.11 is required: newer pydantic breaks gradio's API schema introspection
!pip install -q -r requirements.txt
!pip install -q "pydantic<2.11"


## Get a free Groq API key

1. Go to https://console.groq.com/keys
2. Click "Create API Key" and copy it.
3. Run the next cell **on its own** (not via "Run all") and wait for the input box to appear at the top before pasting. The cell after it does a live test call so you'll know right away if the key works.


In [ ]:
# 4. Configure environment variables
import getpass

while True:
    groq_api_key = getpass.getpass("Enter your Groq API key: ").strip()
    if not groq_api_key:
        print("\u274c Empty input \u2014 the key box may not have been ready. Try again.")
        continue
    break

env_content = f"""GROQ_API_KEY={groq_api_key}
LANGSMITH_API_KEY=your_key_here
LANGCHAIN_PROJECT=supply-chain-risk
LANGCHAIN_TRACING_V2=false
LANGSMITH_TRACING=False
LANGSMITH_ENDPOINT=https://api.smith.langchain.com/
LANGSMITH_PROJECT=supply-chain-risk
"""

with open(".env", "w") as f:
    f.write(env_content)

print(f"\u2713 .env saved (key length: {len(groq_api_key)}). Run the next cell to verify it actually works.")


In [ ]:
# 4b. Sanity-check the key actually works before initializing anything
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv(override=True)
_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
try:
    _resp = _client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "Say OK"}],
    )
    print("\u2713 Key works:", _resp.choices[0].message.content)
except Exception as e:
    print("\u274c Key test failed:", e)
    print("Re-run the previous cell and paste the key again.")


## Write the application source files

Each cell below writes one module of the supply chain risk agent application.


In [ ]:
%%writefile config.py
"""Configuration module for environment variables and settings."""
import os
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables
load_dotenv()

# Groq Configuration (free tier)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in environment variables")

# LangSmith Configuration
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "supply-chain-risk")
LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"

# Database Configuration
DB_PATH = Path("data/supply_chain_risk.db")
DB_PATH.parent.mkdir(exist_ok=True)

# Risk model output (exported from the XGBoost classifier above)
RISK_DATA_PATH = Path("data/risk_predictions.csv")
REGION_SUMMARY_PATH = Path("data/region_risk_summary.csv")
MODE_SUMMARY_PATH = Path("data/shipping_mode_risk_summary.csv")

# RAG Configuration (local embeddings, no API key needed)
RAG_DOCUMENTS_PATH = Path("data/risk_docs")
RAG_DOCUMENTS_PATH.mkdir(exist_ok=True)
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K_RESULTS = 3

# Model Configuration
LLM_MODEL = "llama-3.3-70b-versatile"
TEMPERATURE = 0.3

if __name__ == "__main__":
    # Test configuration
    print("\u2713 Configuration loaded successfully")
    print(f"  LangSmith Project: {LANGSMITH_PROJECT}")
    print(f"  Database Path: {DB_PATH}")
    print(f"  Risk Data Path: {RISK_DATA_PATH}")
    print(f"  LLM Model: {LLM_MODEL}")


In [ ]:
%%writefile db.py
"""Database module for persistent storage with multi-user support."""
import sqlite3
import json
from datetime import datetime
from contextlib import contextmanager
from typing import List, Dict, Optional
import threading
from config import DB_PATH

# Thread-local storage for database connections
_thread_local = threading.local()

class DatabaseManager:
    """Thread-safe database manager for SQLite operations."""

    def __init__(self, db_path: str = DB_PATH):
        self.db_path = str(db_path)
        self._init_db()

    def _get_connection(self):
        """Get thread-local database connection."""
        if not hasattr(_thread_local, "connection"):
            _thread_local.connection = sqlite3.connect(
                self.db_path,
                timeout=30,  # Wait up to 30s for lock
                check_same_thread=False  # We're managing threads manually
            )
            _thread_local.connection.row_factory = sqlite3.Row
        return _thread_local.connection

    @contextmanager
    def get_cursor(self):
        """Context manager for database cursors with automatic commit/rollback."""
        conn = self._get_connection()
        cursor = conn.cursor()
        try:
            yield cursor
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        finally:
            cursor.close()

    def _init_db(self):
        """Initialize database tables if they don't exist."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS conversations (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    user_query TEXT NOT NULL,
                    assistant_response TEXT NOT NULL,
                    agent_used TEXT,
                    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                    metadata TEXT
                )
            """)
            cursor.execute("""
                CREATE INDEX IF NOT EXISTS idx_session_timestamp
                ON conversations(session_id, timestamp)
            """)

    def save_conversation(self, session_id: str, user_query: str,
                         assistant_response: str, agent_used: str = None,
                         metadata: Dict = None):
        """Save a conversation turn to database."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                INSERT INTO conversations
                (session_id, user_query, assistant_response, agent_used, metadata)
                VALUES (?, ?, ?, ?, ?)
            """, (
                session_id,
                user_query,
                assistant_response,
                agent_used,
                json.dumps(metadata) if metadata else None
            ))

    def load_session_history(self, session_id: str, limit: int = 50) -> List[Dict]:
        """Load conversation history for a session."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                SELECT user_query, assistant_response, agent_used, timestamp, metadata
                FROM conversations
                WHERE session_id = ?
                ORDER BY timestamp DESC
                LIMIT ?
            """, (session_id, limit))

            rows = cursor.fetchall()
            return [
                {
                    "user_query": row["user_query"],
                    "assistant_response": row["assistant_response"],
                    "agent_used": row["agent_used"],
                    "timestamp": row["timestamp"],
                    "metadata": json.loads(row["metadata"]) if row["metadata"] else {}
                }
                for row in rows
            ]

    def get_all_sessions(self) -> List[str]:
        """Get all unique session IDs."""
        with self.get_cursor() as cursor:
            cursor.execute("SELECT DISTINCT session_id FROM conversations")
            return [row["session_id"] for row in cursor.fetchall()]

    def delete_session(self, session_id: str):
        """Delete a session and all its conversations."""
        with self.get_cursor() as cursor:
            cursor.execute("DELETE FROM conversations WHERE session_id = ?", (session_id,))


# Global database instance
db_manager = DatabaseManager()

if __name__ == "__main__":
    print("Testing Database Module...")

    test_session = "test_session_123"

    db_manager.save_conversation(
        session_id=test_session,
        user_query="Which orders are at highest risk of late delivery this week?",
        assistant_response="The top risk orders are concentrated in Standard Class shipments to Southeast Asia, with Risk_Probability above 0.8.",
        agent_used="risk_assessment",
        metadata={"test": True}
    )

    history = db_manager.load_session_history(test_session)
    print(f"\u2713 Saved and loaded {len(history)} conversations")

    sessions = db_manager.get_all_sessions()
    print(f"\u2713 Active sessions: {sessions}")

    db_manager.delete_session(test_session)
    print("\u2713 Test session cleaned up")


In [ ]:
%%writefile data_tools.py
"""Structured-data tool module: lets agents query the XGBoost risk predictions directly
instead of relying on the LLM to invent numbers."""
import pandas as pd
from typing import Dict, Any, List, Optional
from config import RISK_DATA_PATH, REGION_SUMMARY_PATH, MODE_SUMMARY_PATH

_risk_df = pd.read_csv(RISK_DATA_PATH, index_col="order_id")
_region_summary = pd.read_csv(REGION_SUMMARY_PATH)
_mode_summary = pd.read_csv(MODE_SUMMARY_PATH)

def get_top_risks(n: int = 10) -> List[Dict[str, Any]]:
    """Return the n highest-risk orders by predicted Risk_Probability."""
    top = _risk_df.sort_values("Risk_Probability", ascending=False).head(n)
    return top.reset_index().to_dict(orient="records")

def get_risk_for_order(order_id: int) -> Optional[Dict[str, Any]]:
    """Return the risk record for a specific order id, if present in the held-out set."""
    if order_id not in _risk_df.index:
        return None
    row = _risk_df.loc[order_id]
    return {"order_id": int(order_id), **row.to_dict()}

def get_region_risk_summary() -> List[Dict[str, Any]]:
    """Return mean predicted risk probability per order region, sorted descending."""
    return _region_summary.to_dict(orient="records")

def get_shipping_mode_risk_summary() -> List[Dict[str, Any]]:
    """Return mean predicted risk probability per shipping mode, sorted descending."""
    return _mode_summary.to_dict(orient="records")

def get_dataset_stats() -> Dict[str, Any]:
    """Return overall summary stats for the held-out risk predictions."""
    return {
        "n_orders": int(len(_risk_df)),
        "mean_risk_probability": float(_risk_df["Risk_Probability"].mean()),
        "actual_late_rate": float(_risk_df["Late_delivery_risk"].mean()),
        "n_regions": int(_region_summary.shape[0]),
        "n_shipping_modes": int(_mode_summary.shape[0]),
    }

if __name__ == "__main__":
    print("Testing Data Tools Module...")
    print(f"\nDataset stats: {get_dataset_stats()}")
    print("\nTop 3 risks:")
    for row in get_top_risks(3):
        print(row)
    print(f"\nRegion risk summary (top 3): {get_region_risk_summary()[:3]}")
    print(f"\nShipping mode risk summary: {get_shipping_mode_risk_summary()}")


In [ ]:
%%writefile rag.py
"""RAG module for document retrieval and context injection.

Uses a local sentence-transformers embedding model rather than a hosted API,
since Groq's free tier does not expose an embeddings endpoint.
"""
import os
from typing import List
from pathlib import Path
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langsmith import traceable
from config import (
    EMBEDDING_MODEL,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    TOP_K_RESULTS,
)

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

class RAGSystem:
    """Lightweight RAG system for supply chain delivery risk documents."""

    def __init__(self, persist_directory: str = "data/faiss_index"):
        self.persist_directory = persist_directory
        self.vector_store = None
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
        )
        self._initialize_vector_store()

    def _initialize_vector_store(self):
        """Initialize FAISS vector store from existing index or create new."""
        if os.path.exists(self.persist_directory):
            try:
                self.vector_store = FAISS.load_local(
                    self.persist_directory,
                    embeddings,
                    allow_dangerous_deserialization=True
                )
                print(f"\u2713 Loaded existing FAISS index from {self.persist_directory}")
            except Exception as e:
                print(f"! Could not load existing index: {e}")
                self.vector_store = None

        if self.vector_store is None:
            self._create_sample_documents()

    def _create_sample_documents(self):
        """Create the supply chain risk knowledge base."""
        documents = [
            Document(
                page_content="""Late Delivery Risk Scoring Guidelines:
                - Risk_Probability is the predicted likelihood (0-1) that an order arrives later than its scheduled delivery date, produced by an XGBoost classifier trained on shipping mode, real vs scheduled transit days, and order region.
                - Orders above 0.7 probability should be treated as high risk and flagged for proactive intervention before dispatch.
                - Orders between 0.4 and 0.7 are medium risk; monitor but no automatic action required.
                - The gap between 'Days for shipping (real)' and 'Days for shipment (scheduled)' is the single strongest signal: any positive gap historically correlates with late delivery.
                - Same Day and First Class shipping modes generally carry lower risk than Standard Class, which has the longest scheduled window and the most variance.""",
                metadata={"category": "risk_definition", "source": "scoring_guidelines"}
            ),
            Document(
                page_content="""Delivery Risk Mitigation Playbook:
                - For high-risk orders (Risk_Probability > 0.7), proactively upgrade shipping mode (e.g. Standard Class -> Second Class or First Class) before dispatch if the margin allows.
                - Notify the customer ahead of time when an order is flagged high risk so expectations are set early, reducing complaint volume even if the order is still late.
                - Build in carrier diversification for regions with chronically high average risk: don't rely on a single carrier or lane.
                - For recurring high-risk lanes, negotiate tighter SLAs with carriers or add a local cross-dock to cut transit variance.
                - Track mitigation effectiveness by comparing actual Late_delivery_risk outcomes before and after an intervention is applied to a lane.""",
                metadata={"category": "mitigation", "source": "mitigation_playbook"}
            ),
            Document(
                page_content="""Regional Risk Distribution Best Practices:
                - Aggregate Risk_Probability by Order Region monthly to spot regions trending up, not just the current snapshot.
                - Regions with average predicted risk meaningfully above the global mean should get a dedicated logistics review: check carrier mix, customs delays, and last-mile partners.
                - Cross-border and long-haul regions typically show higher variance in 'Days for shipping (real)' than domestic regions, which is the main driver of elevated regional risk.
                - When comparing regions, normalize by order volume \u2014 a region with few orders can show a misleadingly high or low average risk.""",
                metadata={"category": "regional_analysis", "source": "regional_guide"}
            ),
            Document(
                page_content="""Shipping Mode Selection Guidance:
                - Standard Class has the longest scheduled window and historically the highest share of late deliveries; reserve it for low-urgency, low-risk orders.
                - First Class and Same Day shipping modes have tighter scheduled windows and lower historical late rates, but cost more \u2014 use selectively for high-value or already-flagged high-risk orders.
                - Second Class sits in between; a reasonable default upgrade path for medium-risk Standard Class orders.
                - Shipping mode alone does not guarantee on-time delivery \u2014 always check the predicted Risk_Probability for the specific order before deciding whether an upgrade is warranted.""",
                metadata={"category": "shipping_modes", "source": "mode_guide"}
            ),
            Document(
                page_content="""Risk Escalation & Reporting Rules:
                - Escalate any single order with Risk_Probability > 0.85 to the logistics duty manager immediately.
                - Daily ops review should include: count of high-risk orders, top 5 riskiest regions, and any shipping mode whose average risk has moved more than 10 percentage points week-over-week.
                - Keep a rolling log of interventions (mode upgrades, carrier swaps, customer notifications) tied to order IDs so mitigation effectiveness can be audited later.
                - Risk_Probability is a model estimate, not a certainty \u2014 always cross-check the top of the risk list against any known carrier outages or weather disruptions before reporting.""",
                metadata={"category": "escalation", "source": "ops_rules"}
            ),
        ]
        self.add_documents(documents)
        print("\u2713 Created sample documents and FAISS index")

    @traceable(name="rag_add_documents", run_type="chain")
    def add_documents(self, documents: List[Document]):
        """Add documents to the vector store."""
        chunks = self.text_splitter.split_documents(documents)
        if self.vector_store is None:
            self.vector_store = FAISS.from_documents(chunks, embeddings)
        else:
            self.vector_store.add_documents(chunks)
        os.makedirs(os.path.dirname(self.persist_directory) or ".", exist_ok=True)
        self.vector_store.save_local(self.persist_directory)

    @traceable(name="rag_retrieve", run_type="retriever")
    def retrieve_context(self, query: str, k: int = TOP_K_RESULTS) -> List[Document]:
        """Retrieve relevant documents for a query."""
        if self.vector_store is None:
            return []
        return self.vector_store.similarity_search(query, k=k)

    @traceable(name="rag_get_context", run_type="chain")
    def get_context_string(self, query: str) -> str:
        """Get context as a formatted string for prompt injection."""
        docs = self.retrieve_context(query)
        if not docs:
            return "No relevant documents found."
        context_parts = []
        for i, doc in enumerate(docs, 1):
            source = doc.metadata.get("source", "Unknown")
            category = doc.metadata.get("category", "General")
            context_parts.append(f"[Document {i} - {category} ({source})]:\n{doc.page_content}\n")
        return "\n".join(context_parts)


# Global RAG instance
rag_system = RAGSystem()

if __name__ == "__main__":
    print("Testing RAG Module...")
    test_queries = [
        "What does a Risk_Probability above 0.7 mean?",
        "How can we reduce late delivery risk for Standard Class shipments?",
        "How should we compare risk across regions?",
    ]
    for query in test_queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        context = rag_system.get_context_string(query)
        print(f"Retrieved Context:\n{context}")


In [ ]:
%%writefile router.py
"""Router module for classifying user intent in supply chain risk queries."""
import json
from typing import Dict, Any
from groq import Groq
from langsmith import traceable
from config import GROQ_API_KEY, LLM_MODEL

client = Groq(api_key=GROQ_API_KEY)

@traceable(name="router_classification", run_type="chain")
def classify_intent(query: str) -> Dict[str, Any]:
    """
    Classify user query into one of the supply chain risk categories.

    Args:
        query: User's question

    Returns:
        Dictionary with classification result and confidence
    """
    system_prompt = """You are an intent classifier for a Supply Chain Delivery Risk System.
    Classify the user's query into one of these categories:

    1. risk_assessment - Questions about a specific order's risk score, the current riskiest orders, or what a Risk_Probability value means
    2. mitigation_strategy - Questions about how to reduce, prevent, or respond to late delivery risk
    3. regional_analysis - Questions comparing risk across regions, shipping modes, or lanes
    4. general - General questions not specific to the above categories

    Respond with JSON format: {"category": "category_name", "confidence": 0.0-1.0, "reasoning": "brief explanation"}
    """

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ]
    )

    try:
        result = json.loads(response.choices[0].message.content)
    except Exception:
        result = {
            "category": "general",
            "confidence": 0.5,
            "reasoning": "Failed to parse response"
        }

    result["usage"] = response.usage.model_dump() if response.usage else None
    return result

@traceable(name="router_decision", run_type="chain")
def route_query(query: str) -> str:
    """
    Route query to appropriate agent based on intent.

    Args:
        query: User's question

    Returns:
        Agent name to route to
    """
    classification = classify_intent(query)
    return classification.get("category", "general")

if __name__ == "__main__":
    print("Testing Router Module...")

    test_queries = [
        "What's the risk score for order 12345?",
        "How can we cut down on late deliveries for Standard Class?",
        "Which region has the worst delivery risk this month?",
        "Tell me about how this risk model works"
    ]

    for query in test_queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        classification = classify_intent(query)
        print(f"Classification: {classification}")
        print(f"Routed to: {route_query(query)}")


In [ ]:
%%writefile agents/__init__.py



In [ ]:
%%writefile agents/risk_assessment_agent.py
"""Risk Assessment Agent for supply chain delivery risk."""
import re
from typing import Dict, Any
from groq import Groq
from langsmith import traceable
from config import GROQ_API_KEY, LLM_MODEL, TEMPERATURE
from data_tools import get_risk_for_order, get_top_risks

client = Groq(api_key=GROQ_API_KEY)

@traceable(name="risk_assessment_agent", run_type="chain")
def risk_assessment_agent(query: str, context: str = None) -> Dict[str, Any]:
    """
    Risk Assessment Agent - Explains predicted late-delivery risk for orders.

    Looks for an explicit order id in the query and pulls its real predicted
    Risk_Probability from the XGBoost model's held-out predictions; otherwise
    falls back to the current top-risk orders as grounding data.

    Args:
        query: User query about order risk
        context: Retrieved RAG context

    Returns:
        Dictionary with response and metadata
    """
    order_match = re.search(r"\border\s*#?\s*(\d+)", query, re.IGNORECASE)
    tool_result = None
    if order_match:
        tool_result = get_risk_for_order(int(order_match.group(1)))
    if tool_result is None:
        tool_result = {"top_risk_orders": get_top_risks(5)}

    system_prompt = """You are a Supply Chain Delivery Risk Assessment Expert. Your role is to:
    - Explain what a given Risk_Probability score means for a specific order or set of orders
    - Identify which factors (shipping mode, scheduled vs real transit days, region) likely drive the score
    - Be precise and quote the actual numbers provided in the tool data, never invent figures
    - Recommend whether the order needs proactive intervention based on the risk thresholds in the guidelines

    Use the provided tool data (real model output) and context documents to ground your answer."""

    user_prompt = f"""Tool data (live model output):
    {tool_result}

    Context from supply chain risk guidelines:
    {context if context else "No specific context provided."}

    User Question: {query}

    Please provide a grounded risk assessment."""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return {
        "agent": "risk_assessment",
        "response": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else None
    }

if __name__ == "__main__":
    print("Testing Risk Assessment Agent...")

    test_query = "What are the riskiest orders right now and why?"
    result = risk_assessment_agent(test_query, "Orders above 0.7 probability are high risk.")

    print(f"\nQuery: {test_query}")
    print(f"\nResponse: {result['response']}")
    print(f"\nToken Usage: {result['usage']}")


In [ ]:
%%writefile agents/mitigation_agent.py
"""Mitigation Strategy Agent for supply chain delivery risk."""
from typing import Dict, Any
from groq import Groq
from langsmith import traceable
from config import GROQ_API_KEY, LLM_MODEL, TEMPERATURE
from data_tools import get_top_risks

client = Groq(api_key=GROQ_API_KEY)

@traceable(name="mitigation_agent", run_type="chain")
def mitigation_agent(query: str, context: str = None) -> Dict[str, Any]:
    """
    Mitigation Agent - Recommends ways to reduce or respond to late delivery risk.

    Args:
        query: User query about mitigation strategy
        context: Retrieved RAG context

    Returns:
        Dictionary with response and metadata
    """
    tool_result = {"current_top_risk_orders": get_top_risks(5)}

    system_prompt = """You are a Supply Chain Delivery Risk Mitigation Expert. Your role is to:
    - Recommend concrete mitigation actions (shipping mode upgrades, carrier diversification, customer notification, safety stock)
    - Tie recommendations to the actual high-risk orders provided in the tool data when relevant
    - Prioritize actions by expected impact versus cost
    - Reference escalation thresholds from the guidelines when an order is severe enough to warrant manager attention

    Use the provided tool data and context documents to ground your recommendations."""

    user_prompt = f"""Tool data (current highest-risk orders):
    {tool_result}

    Context from supply chain risk guidelines:
    {context if context else "No specific context provided."}

    User Question: {query}

    Please provide a prioritized mitigation plan."""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return {
        "agent": "mitigation_strategy",
        "response": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else None
    }

if __name__ == "__main__":
    print("Testing Mitigation Agent...")

    test_query = "How can we reduce late delivery risk for Standard Class shipments?"
    result = mitigation_agent(test_query, "Upgrade high-risk Standard Class orders to Second Class before dispatch.")

    print(f"\nQuery: {test_query}")
    print(f"\nResponse: {result['response']}")
    print(f"\nToken Usage: {result['usage']}")


In [ ]:
%%writefile agents/regional_agent.py
"""Regional Analysis Agent for supply chain delivery risk."""
from typing import Dict, Any
from groq import Groq
from langsmith import traceable
from config import GROQ_API_KEY, LLM_MODEL, TEMPERATURE
from data_tools import get_region_risk_summary, get_shipping_mode_risk_summary

client = Groq(api_key=GROQ_API_KEY)

@traceable(name="regional_agent", run_type="chain")
def regional_agent(query: str, context: str = None) -> Dict[str, Any]:
    """
    Regional Analysis Agent - Compares delivery risk across regions and shipping modes.

    Args:
        query: User query about regional or shipping mode comparisons
        context: Retrieved RAG context

    Returns:
        Dictionary with response and metadata
    """
    tool_result = {
        "region_risk_summary": get_region_risk_summary(),
        "shipping_mode_risk_summary": get_shipping_mode_risk_summary(),
    }

    system_prompt = """You are a Supply Chain Regional Risk Analyst. Your role is to:
    - Compare average predicted delivery risk across regions and shipping modes using the real aggregates in the tool data
    - Identify which regions or modes are outliers relative to the overall average
    - Caveat conclusions drawn from regions with low order volume
    - Recommend where a dedicated logistics review is warranted

    Use the provided tool data (real model aggregates) and context documents to ground your answer."""

    user_prompt = f"""Tool data (real aggregated model output):
    {tool_result}

    Context from supply chain risk guidelines:
    {context if context else "No specific context provided."}

    User Question: {query}

    Please provide a grounded regional/shipping-mode comparison."""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return {
        "agent": "regional_analysis",
        "response": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else None
    }

if __name__ == "__main__":
    print("Testing Regional Analysis Agent...")

    test_query = "Which regions have the highest average delivery risk?"
    result = regional_agent(test_query, "Normalize by order volume before comparing regions.")

    print(f"\nQuery: {test_query}")
    print(f"\nResponse: {result['response']}")
    print(f"\nToken Usage: {result['usage']}")


In [ ]:
%%writefile graph.py
"""LangGraph workflow for the Supply Chain Risk Assistant."""
from typing import Dict, Any, Literal
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict
from langsmith import traceable

from agents.risk_assessment_agent import risk_assessment_agent
from agents.mitigation_agent import mitigation_agent
from agents.regional_agent import regional_agent
from router import classify_intent
from rag import rag_system
from db import db_manager
from groq import Groq
from config import GROQ_API_KEY, LLM_MODEL

client = Groq(api_key=GROQ_API_KEY)

# Define state schema with conversation context
class AgentState(TypedDict):
    """State for the LangGraph agent workflow."""
    query: str
    session_id: str
    conversation_context: str
    intent: str
    intent_confidence: float
    intent_reasoning: str
    rag_context: str
    agent_response: str
    agent_used: str
    usage: Dict
    error: str

@traceable(name="inject_context", run_type="chain")
def inject_context_node(state: AgentState) -> AgentState:
    """Node to inject RAG context based on intent."""
    try:
        query = state["query"]
        context = state.get("conversation_context", "")

        enhanced_query = query
        if context:
            context_lines = context.split("\n")
            last_user_msg = None
            for line in reversed(context_lines):
                if line.startswith("User:"):
                    last_user_msg = line.replace("User:", "").strip()
                    break
            if last_user_msg:
                enhanced_query = f"Previous question: {last_user_msg}\nCurrent question: {query}"

        state["rag_context"] = rag_system.get_context_string(enhanced_query)
    except Exception as e:
        state["rag_context"] = ""
        state["error"] = f"RAG error: {str(e)}"

    return state

@traceable(name="router_node", run_type="chain")
def router_node(state: AgentState) -> AgentState:
    """Node to classify intent with conversation context."""
    try:
        query = state["query"]
        context = state.get("conversation_context", "")

        enhanced_query = query
        if context:
            context_lines = context.split("\n")
            recent_exchanges = context_lines[-4:] if len(context_lines) > 4 else context_lines
            enhanced_query = f"""Previous conversation:
{chr(10).join(recent_exchanges)}

Current question: {query}"""

        classification = classify_intent(enhanced_query)
        state["intent"] = classification.get("category", "general")
        state["intent_confidence"] = classification.get("confidence", 0.0)
        state["intent_reasoning"] = classification.get("reasoning", "")
        state["usage"] = classification.get("usage")
    except Exception as e:
        state["intent"] = "general"
        state["error"] = f"Router error: {str(e)}"

    return state

def _build_enhanced_context(state: AgentState) -> str:
    rag_context = state["rag_context"]
    conversation_context = state.get("conversation_context", "")
    if conversation_context:
        return f"""Previous conversation context:
{conversation_context}

Relevant documentation:
{rag_context}"""
    return rag_context

@traceable(name="risk_assessment_agent_node", run_type="chain")
def risk_agent_node(state: AgentState) -> AgentState:
    """Node for the risk assessment agent."""
    result = risk_assessment_agent(state["query"], _build_enhanced_context(state))
    state["agent_response"] = result["response"]
    state["agent_used"] = result["agent"]
    state["usage"] = result["usage"]
    return state

@traceable(name="mitigation_agent_node", run_type="chain")
def mitigation_agent_node(state: AgentState) -> AgentState:
    """Node for the mitigation strategy agent."""
    result = mitigation_agent(state["query"], _build_enhanced_context(state))
    state["agent_response"] = result["response"]
    state["agent_used"] = result["agent"]
    state["usage"] = result["usage"]
    return state

@traceable(name="regional_agent_node", run_type="chain")
def regional_agent_node(state: AgentState) -> AgentState:
    """Node for the regional analysis agent."""
    result = regional_agent(state["query"], _build_enhanced_context(state))
    state["agent_response"] = result["response"]
    state["agent_used"] = result["agent"]
    state["usage"] = result["usage"]
    return state

@traceable(name="general_agent_node", run_type="chain")
def general_agent_node(state: AgentState) -> AgentState:
    """Node for handling general queries with conversation context."""
    query = state["query"]
    context = state.get("conversation_context", "")

    messages = [
        {"role": "system", "content": "You are a helpful assistant for supply chain delivery risk. Provide general information and guide users to specific agents for detailed queries."}
    ]
    if context:
        messages.append({"role": "system", "content": f"Previous conversation:\n{context}"})
    messages.append({"role": "user", "content": query})

    response = client.chat.completions.create(model=LLM_MODEL, messages=messages)

    state["agent_response"] = response.choices[0].message.content
    state["agent_used"] = "general"
    state["usage"] = response.usage.model_dump() if response.usage else None
    return state

@traceable(name="save_to_db", run_type="chain")
def save_to_db_node(state: AgentState) -> AgentState:
    """Node to save conversation to database."""
    try:
        db_manager.save_conversation(
            session_id=state["session_id"],
            user_query=state["query"],
            assistant_response=state["agent_response"],
            agent_used=state["agent_used"],
            metadata={
                "intent": state["intent"],
                "confidence": state["intent_confidence"],
                "conversation_context": state.get("conversation_context", ""),
                "usage": state["usage"]
            }
        )
    except Exception as e:
        state["error"] = f"Database error: {str(e)}"

    return state

def should_continue(state: AgentState) -> Literal["risk", "mitigation", "regional", "general", END]:
    """Conditional edge to route to appropriate agent."""
    if state.get("error"):
        return END

    intent = state.get("intent", "general")

    if intent == "risk_assessment":
        return "risk"
    elif intent == "mitigation_strategy":
        return "mitigation"
    elif intent == "regional_analysis":
        return "regional"
    else:
        return "general"

def build_risk_assistant_graph():
    """Build and compile the LangGraph workflow."""
    workflow = StateGraph(AgentState)

    workflow.add_node("router", router_node)
    workflow.add_node("inject_context", inject_context_node)
    workflow.add_node("risk", risk_agent_node)
    workflow.add_node("mitigation", mitigation_agent_node)
    workflow.add_node("regional", regional_agent_node)
    workflow.add_node("general", general_agent_node)
    workflow.add_node("save_db", save_to_db_node)

    workflow.set_entry_point("router")
    workflow.add_edge("router", "inject_context")

    workflow.add_conditional_edges(
        "inject_context",
        should_continue,
        {
            "risk": "risk",
            "mitigation": "mitigation",
            "regional": "regional",
            "general": "general",
            END: END
        }
    )

    workflow.add_edge("risk", "save_db")
    workflow.add_edge("mitigation", "save_db")
    workflow.add_edge("regional", "save_db")
    workflow.add_edge("general", "save_db")
    workflow.add_edge("save_db", END)

    return workflow.compile()

# Create global graph instance
risk_assistant_graph = build_risk_assistant_graph()

if __name__ == "__main__":

    ascii_data = risk_assistant_graph.get_graph().draw_ascii()
    print(ascii_data)

    test_queries = [
        ("test_session_1", "What are the riskiest orders right now?"),
        ("test_session_1", "How should we mitigate that?"),  # has context
        ("test_session_2", "Which region has the highest average delivery risk?"),
        ("test_session_3", "What does Risk_Probability actually measure?")
    ]

    for session_id, query in test_queries:
        print(f"\nProcessing: {query}")
        print("-" * 50)

        initial_state = {
            "query": query,
            "session_id": session_id,
            "conversation_context": "",
            "intent": "",
            "intent_confidence": 0.0,
            "intent_reasoning": "",
            "rag_context": "",
            "agent_response": "",
            "agent_used": "",
            "usage": {},
            "error": ""
        }

        result = risk_assistant_graph.invoke(initial_state)

        print(f"Intent: {result['intent']} (confidence: {result['intent_confidence']:.2f})")
        print(f"Agent Used: {result['agent_used']}")
        print(f"Response: {result['agent_response'][:100]}...")
        print("\u2713 Graph execution complete")


In [ ]:
%%writefile gradio_ui.py
"""
Supply Chain Risk Assistant - UI
Run with: python gradio_ui.py
"""
import uuid
import gradio as gr
from graph import risk_assistant_graph
from db import db_manager

# Store session per user
sessions = {}

def process_query(query, history, session_id):
    """Process a single query and return response."""

    if not session_id:
        session_id = str(uuid.uuid4())
        sessions[session_id] = []

    conversation_context = ""
    if history:
        context_parts = []
        for msg in history:
            if isinstance(msg, dict) and "role" in msg and "content" in msg:
                role = "User" if msg["role"] == "user" else "Assistant"
                context_parts.append(f"{role}: {msg['content']}")
        conversation_context = "\n".join(context_parts[-6:])

    state = {
        "query": query,
        "session_id": session_id,
        "conversation_context": conversation_context,
        "intent": "",
        "intent_confidence": 0.0,
        "intent_reasoning": "",
        "rag_context": "",
        "agent_response": "",
        "agent_used": "",
        "usage": {},
        "error": ""
    }

    try:
        result = risk_assistant_graph.invoke(state)

        if result.get("error"):
            response = f"\u274c Error: {result['error']}"
        else:
            agent = result["agent_used"]
            intent = result["intent"]
            confidence = result["intent_confidence"]
            answer = result["agent_response"]

            response = f"""**Agent:** {agent}
**Intent:** {intent} ({confidence:.2f})

{answer}"""

            if result.get("usage"):
                tokens = result["usage"].get("total_tokens", 0)
                response += f"\n\n---\n*Tokens: {tokens}*"

        if history is None:
            history = []

        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": response})

        if session_id in sessions:
            sessions[session_id] = history

        return "", history, session_id

    except Exception as e:
        error_msg = f"\u274c Error: {str(e)}"
        if history is None:
            history = []

        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": error_msg})

        if session_id in sessions:
            sessions[session_id] = history

        return "", history, session_id

def clear_chat(session_id):
    """Clear chat history for a session."""
    if session_id in sessions:
        sessions[session_id] = []
    return [], session_id

def view_history(session_id):
    """View full session history from DB."""
    if not session_id:
        return "No active session"

    history = db_manager.load_session_history(session_id)
    if not history:
        return "No history found"

    output = f"## Session History: {session_id}\n\n"
    for msg in history:
        output += f"**You:** {msg['user_query']}\n\n"
        output += f"**AI ({msg['agent_used']}):** {msg['assistant_response']}\n\n"
        output += "---\n\n"

    return output

def create_session():
    """Create a new session and return session ID."""
    session_id = str(uuid.uuid4())
    sessions[session_id] = []
    return session_id

# Create Gradio interface
with gr.Blocks(title="Supply Chain Risk Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # \U0001F69A Supply Chain Risk Assistant
    Ask about specific order risk, mitigation strategies, or regional/shipping-mode risk comparisons.
    """)

    session_state = gr.State(create_session)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Conversation",
                height=500,
                type="messages"
            )
            msg = gr.Textbox(label="Your Question", placeholder="e.g., What are the riskiest orders right now?")

            with gr.Row():
                submit = gr.Button("Send", variant="primary")
                clear = gr.Button("Clear Chat")

        with gr.Column(scale=1):
            gr.Markdown("### Session Info")
            session_id_display = gr.Textbox(
                label="Session ID",
                value="",
                interactive=False
            )

            new_session_btn = gr.Button("\U0001F195 New Session", variant="secondary")

            gr.Markdown("### Sample Queries")
            sample_queries = gr.Dataset(
                components=[msg],
                samples=[
                    ["What are the riskiest orders right now?"],
                    ["How can we reduce late delivery risk for Standard Class shipments?"],
                    ["Which regions have the highest average delivery risk?"],
                    ["What does a Risk_Probability of 0.8 mean?"],
                    ["Compare risk across shipping modes"]
                ],
                label="Click to try"
            )

            history_btn = gr.Button("View Full History")
            history_output = gr.Markdown()

    def respond(message, chat_history, session_id):
        if not message:
            return "", chat_history, session_id
        if chat_history is None:
            chat_history = []
        return process_query(message, chat_history, session_id)

    def update_session_id(session_id):
        return session_id if session_id else "No active session"

    def new_session():
        new_id = str(uuid.uuid4())
        sessions[new_id] = []
        return [], new_id, new_id

    def clear_chat_handler(session_id):
        if session_id in sessions:
            sessions[session_id] = []
        return [], session_id

    submit.click(respond, [msg, chatbot, session_state], [msg, chatbot, session_state])
    msg.submit(respond, [msg, chatbot, session_state], [msg, chatbot, session_state])

    sample_queries.click(lambda x: x[0], [sample_queries], [msg])

    clear.click(clear_chat_handler, [session_state], [chatbot, msg])

    new_session_btn.click(new_session, None, [chatbot, session_state, session_id_display])

    session_state.change(update_session_id, [session_state], [session_id_display])

    history_btn.click(view_history, [session_state], [history_output])

    demo.load(lambda s: s, [session_state], [session_id_display])

if __name__ == "__main__":
    demo.launch(
        share=False,
        server_name="127.0.0.1",
        server_port=7860
    )


In [ ]:
# 5. Initialize the database, data tools, and RAG (FAISS) index
!python db.py
!python data_tools.py
!python rag.py


In [ ]:
# 6. Launch the Gradio app
# share=True is required on Colab since 127.0.0.1 isn't reachable from your browser —
# Gradio will print a public *.gradio.live link instead.
from gradio_ui import demo

demo.launch(share=True)


## Notes

- **Rate limits:** Groq's free tier for `llama-3.3-70b-versatile` is generous but still rate-limited; space out test queries by a few seconds if you see 429s.
- **Embeddings run locally:** the RAG index uses a local `sentence-transformers` model, so no embedding API key is needed \u2014 only `GROQ_API_KEY` for the chat model.
- **Restarting:** if you restart the Colab runtime, re-run all cells from the top \u2014 the working directory, `.env`, database, exported risk data, and FAISS index are all wiped.
- **Stopping the app:** Use Runtime \u2192 Interrupt execution to stop the Gradio server.
- **Updating this notebook:** since this app is embedded directly rather than imported from a separate module, any future change to the agent logic needs to be re-applied here too.
